<a href="https://colab.research.google.com/github/lilianeuwase2/math_forML-Formative3/blob/main/ProbabilityDistributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import csv
import numpy as np

# df = pd.read_csv("GaltonFamilies.csv")
mothers = []
children = []
seen_families = set()
with open('GaltonFamilies.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        children.append(float(row['childHeight']))
        if row["family"] not in seen_families:
          seen_families.add(row["family"])
          mothers.append(float(row['mother']))
children = np.array(children)
mothers = np.array(mothers)

# #use drop duplicates on the mothers column so that we have just one mother per family
# mothers = df.drop_duplicates('family')['mother'].values
# children = df['childHeight'].values

heights = np.concatenate([mothers, children])
print(len(mothers), len(children), len(heights))

205 934 1139


Gaussian Function


In [14]:
def gaussian(heights, mu, var):
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-(heights- mu) ** 2 / (2 * var))

Log-Likelihood

In [15]:
def log_likelihood(heights, mu1, mu2, var1, var2, pi1, pi2):
    total = pi1 * gaussian(heights, mu1, var1) + pi2 * gaussian(heights, mu2, var2)
    return np.sum(np.log(total))

E-step separate

In [16]:
def e_step(heights, mu1, mu2, var1, var2, pi1, pi2):
    score1 = pi1 * gaussian(heights, mu1, var1)
    score2 = pi2 * gaussian(heights, mu2, var2)
    total = score1 + score2
    return score1 / total, score2 / total

M-step separate

In [17]:
def m_step(heights, prob1, prob2):
    N = len(heights)
    pi1 = prob1.sum() / N
    pi2 = prob2.sum() / N
    mu1 = np.sum(prob1 * heights) / prob1.sum()
    mu2 = np.sum(prob2 * heights) / prob2.sum()
    var1 = np.sum(prob1 * (heights - mu1) ** 2) / prob1.sum()
    var2 = np.sum(prob2 * (heights - mu2) ** 2) / prob2.sum()
    return mu1, mu2, var1, var2, pi1, pi2

the Loop

In [18]:
def run_em(heights, n_iter):
    params = (heights.mean() - heights.std(), heights.mean() + heights.std(), heights.var(), heights.var(), 0.5, 0.5)
    history = [[0, *params, log_likelihood(heights, *params)]]

    for it in range(1, n_iter + 1):
        prob1, prob2 = e_step(heights, *params)
        params = m_step(heights, prob1, prob2)
        history.append([it, *params, log_likelihood(heights, *params)])

    return history, params

history, params = run_em(heights, 2)
print("iter", "mu1", "mu2", "var1", "var2", "pi1", "pi2", "log_lik")
for h in history:
  print(f"{h[0]:>4} " + " ".join(f"{v:9.4f}" for v in h[1:]))

iter mu1 mu2 var1 var2 pi1 pi2 log_lik
   0   62.7052   69.7989   12.5802   12.5802    0.5000    0.5000 -3190.3325
   1   64.0693   68.5283    6.5015    8.7695    0.5105    0.4895 -3041.3709
   2   64.0394   68.5861    6.0192    8.8886    0.5133    0.4867 -3039.7574
